In [6]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

In [8]:
# Loading file
Template_ZCTA_County22 = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/Template_ZCTA_County22.csv'))

In [9]:
# Defining the election month starting November 1st, 2022
election_date = pd.Timestamp('2022-11-01')

# Extracting only the needed columns and ensuring 'Month' is in the correct format
county_data = Template_ZCTA_County22[['ZCTA', 'admin2', 'Month']]
county_data['Month'] = pd.to_datetime(county_data['Month'], errors='coerce')

# Function to calculate months since/until election
def calculate_month_difference(row, election_date, before=True):
    delta = (row['Month'].year - election_date.year) * 12 + (row['Month'].month - election_date.month)
    return -delta if before else delta

# Calculating months until and since the election
county_data['months_until_election'] = county_data.apply(calculate_month_difference, args=(election_date, True), axis=1)
county_data['months_since_election'] = county_data.apply(calculate_month_difference, args=(election_date, False), axis=1)

# Formatting the 'Month' column
county_data['Month'] = county_data['Month'].dt.strftime('%Y-%m')

# Adding a binary indicator for the election month (November 2022)
county_data['election_month'] = county_data['Month'].apply(lambda x: 1 if x == '2022-11' else 0)


In [10]:
#save as csv
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "Election_Feature_Months.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

county_data.to_csv(output_path_1, index=False)